In [ ]:
# %pip install datasets open_clip_torch ftfy regex tqdm scikit-learn pandas pillow torch torchvision

In [ ]:
import torch
import numpy as np
import pandas as pd
import re
from PIL import Image
from datasets import load_dataset
from tqdm import tqdm
from sklearn.metrics import accuracy_score, f1_score, classification_report
import open_clip

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("using:", device)

In [ ]:
dataset = load_dataset("flwrlabs/ucf101")
dataset = dataset["test"]

print(dataset)
print(dataset.column_names)
print(dataset.features)
print(dataset[0])

In [ ]:
def find_column(column_names, candidates):
    for name in candidates:
        if name in column_names:
            return name
    return None


image_col = find_column(dataset.column_names, ["image", "img", "frame"])
label_col = find_column(dataset.column_names, ["label", "class", "target"])

print("image column:", image_col)
print("label column:", label_col)

if image_col is None:
    raise ValueError("Cannot find image column. Please check dataset.column_names.")

if label_col is None:
    raise ValueError("Cannot find label column. Please check dataset.column_names.")

In [ ]:
label_feature = dataset.features[label_col]

if hasattr(label_feature, "names") and label_feature.names is not None:
    label_names = label_feature.names
else:
    unique_labels = sorted(set(dataset[label_col]))
    label_names = [str(label) for label in unique_labels]

num_classes = len(label_names)

print("number of classes:", num_classes)
print(label_names[:20])

In [ ]:
def clean_label_name(name):
    name = str(name)
    name = name.replace("_", " ")
    name = re.sub(r"([a-z])([A-Z])", r"\1 \2", name)
    return name.lower()


prompts = [
    f"a video of a person doing {clean_label_name(name)}"
    for name in label_names
]

print("number of prompts:", len(prompts))
print(prompts[:20])

In [ ]:
model, _, preprocess = open_clip.create_model_and_transforms(
    "ViT-B-32",
    pretrained="openai"
)

tokenizer = open_clip.get_tokenizer("ViT-B-32")

model = model.to(device)
model.eval()

with torch.no_grad():
    text_tokens = tokenizer(prompts).to(device)
    text_features = model.encode_text(text_tokens)
    text_features = text_features / text_features.norm(dim=-1, keepdim=True)

print("text features shape:", text_features.shape)

In [ ]:
def get_pil_image(example):
    img = example[image_col]

    if isinstance(img, Image.Image):
        return img.convert("RGB")

    if isinstance(img, dict) and "bytes" in img:
        from io import BytesIO
        return Image.open(BytesIO(img["bytes"])).convert("RGB")

    if isinstance(img, dict) and "path" in img:
        return Image.open(img["path"]).convert("RGB")

    if isinstance(img, str):
        return Image.open(img).convert("RGB")

    raise TypeError(f"Unsupported image type: {type(img)}")


def get_label_id(example):
    label = example[label_col]

    if isinstance(label, int):
        return label

    if label in label_names:
        return label_names.index(label)

    return int(label)

In [ ]:
BATCH_SIZE = 32

y_true = []
y_pred = []

batch_images = []
batch_labels = []

with torch.no_grad():
    for example in tqdm(dataset):
        img = get_pil_image(example)
        label = get_label_id(example)

        batch_images.append(preprocess(img))
        batch_labels.append(label)

        if len(batch_images) == BATCH_SIZE:
            image_tensor = torch.stack(batch_images).to(device)

            image_features = model.encode_image(image_tensor)
            image_features = image_features / image_features.norm(dim=-1, keepdim=True)

            logits = 100.0 * image_features @ text_features.T
            preds = logits.argmax(dim=-1).cpu().numpy()

            y_true.extend(batch_labels)
            y_pred.extend(preds)

            batch_images = []
            batch_labels = []

    if len(batch_images) > 0:
        image_tensor = torch.stack(batch_images).to(device)

        image_features = model.encode_image(image_tensor)
        image_features = image_features / image_features.norm(dim=-1, keepdim=True)

        logits = 100.0 * image_features @ text_features.T
        preds = logits.argmax(dim=-1).cpu().numpy()

        y_true.extend(batch_labels)
        y_pred.extend(preds)

print("number of evaluated examples:", len(y_true))

In [ ]:
accuracy = accuracy_score(y_true, y_pred)
macro_f1 = f1_score(y_true, y_pred, average="macro")

print("Accuracy:", accuracy)
print("Macro F1:", macro_f1)

print(classification_report(
    y_true,
    y_pred,
    labels=list(range(num_classes)),
    target_names=label_names,
    zero_division=0
))

In [ ]:
results_df = pd.DataFrame({
    "true_label_id": y_true,
    "pred_label_id": y_pred,
    "true_label": [label_names[i] for i in y_true],
    "pred_label": [label_names[i] for i in y_pred]
})

results_df["correct"] = results_df["true_label_id"] == results_df["pred_label_id"]

results_df.to_csv("clip_ucf101_full_predictions.csv", index=False)

print("saved to clip_ucf101_full_predictions.csv")
results_df.head()